In [0]:
from pyspark.sql.functions import (
    col,
    current_timestamp,
    current_date,
    to_date,
    split,
    trim,
    transform,
    array_distinct,
    when,
    length,
    row_number
)

from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
spark.sql("USE CATALOG skitkartev")

spark.sql("""
CREATE SCHEMA IF NOT EXISTS silver
""")

DataFrame[]

In [0]:
bronze_df = spark.table("skitkartev.bronze.jobs")

In [0]:
silver_df = (
    bronze_df
    .withColumn("DatePosted", to_date(col("DatePosted")))
    .withColumn("ApplicationDeadline", to_date(col("ApplicationDeadline")))
    .withColumn("UpdatedTimestamp", col("UpdatedTimestamp").cast("timestamp"))
)

In [0]:
silver_df = silver_df.filter(
    col("JobID").isNotNull() &
    col("JobTitle").isNotNull() &
    col("CompanyName").isNotNull()
)

In [0]:
silver_df = silver_df.filter(
    (col("DatePosted") <= current_date()) &
    (col("ApplicationDeadline") >= col("DatePosted"))
)

In [0]:
silver_df = (
    silver_df
    .withColumn("SalaryMin", col("SalaryMin").cast("double"))
    .withColumn("SalaryMax", col("SalaryMax").cast("double"))
)

In [0]:
silver_df = silver_df.filter(
    col("SalaryMin").isNotNull() &
    col("SalaryMax").isNotNull() &
    (col("SalaryMin") <= col("SalaryMax"))
)

In [0]:
silver_df = silver_df.withColumn(
    "RequiredSkills",
    array_distinct(
        transform(
            split(col("RequiredSkills"), ","),
            lambda x: trim(x)
        )
    )
)

In [0]:
silver_df = silver_df.withColumn(
    "PreferredSkills",
    array_distinct(
        transform(
            split(col("PreferredSkills"), ","),
            lambda x: trim(x)
        )
    )
)

In [0]:
silver_df = silver_df.filter(

    (length(col("JobTitle")) <= 150) &

    (length(col("CompanyName")) <= 200) &

    (
        col("RecruiterEmail").isNull()
        |
        (length(col("RecruiterEmail")) <= 255)
    )

)

In [0]:
window_spec = Window.partitionBy("JobID").orderBy(
    col("UpdatedTimestamp").desc()
)

silver_df = (
    silver_df
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

In [0]:
silver_df = silver_df.withColumn(
    "SilverProcessedTimestamp",
    current_timestamp()
)

In [0]:
if not spark.catalog.tableExists("skitkartev.silver.jobs"):

    (
        silver_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("skitkartev.silver.jobs")
    )
else:

    delta_table = DeltaTable.forName(
        spark,
        "skitkartev.silver.jobs"
    )

    (
        delta_table.alias("target")
        .merge(
            silver_df.alias("source"),
            """
            target.JobID = source.JobID
            """
        )
        .whenMatchedUpdate(
            condition="""
                source.UpdatedTimestamp > target.UpdatedTimestamp
            """,
            set={
                column: f"source.{column}"
                for column in silver_df.columns
            }
        )
        .whenNotMatchedInsert(
            values={
                column: f"source.{column}"
                for column in silver_df.columns
            }
        )
        .execute()
    )

In [0]:
%sql
SELECT COUNT(*) AS SilverRows
FROM skitkartev.silver.jobs;

SilverRows
502496


In [0]:
%sql
SELECT JobID, COUNT(*) AS Cnt
FROM skitkartev.silver.jobs
GROUP BY JobID
HAVING COUNT(*) > 1;

JobID,Cnt
